In [1]:
# =============================================================================
#  Hardware Parameter Sweep — IBM Quantum Task 6
#  Coarse (θ,φ) grid on ibm_torino: F_msg(θ,φ) and p_succ(θ,φ)
#
#  Circuit:  Deterministic decoder (Implementation A structure) with
#            V(θ,φ) = Rz(φ)·Rx(θ) on register C before scrambling.
#            Output qubit Y measured directly in Z/X/Y Pauli bases.
#
#  Grid:     6×8 = 48 (θ,φ) points, θ ∈ [0,π], φ ∈ [0,2π]
#            3 tomography circuits per point → 144 circuits total
#            2000 shots per circuit → 288k total shots
#
#  Outputs:  hw_sweep_results.json
#            (pass to plot_hw_sweep.py for all figures)
# =============================================================================

import numpy as np
import warnings, json, datetime
warnings.filterwarnings("ignore")
from collections import Counter
from scipy.stats import beta as beta_dist
from math import pi

from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity, DensityMatrix
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

# =============================================================================
#  CONFIG
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 2000          # per tomography circuit
N_BOOTSTRAP = 1000
SEED        = 42
RNG         = np.random.default_rng(SEED)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195

# Coarse sweep grid — chosen to capture both robust and fragile regions
N_THETA     = 6
N_PHI       = 8
THETAS      = np.linspace(0, pi, N_THETA)          # θ ∈ [0, π]
PHIS        = np.linspace(0, 2*pi, N_PHI)           # φ ∈ [0, 2π]

# =============================================================================
#  STEP 1 — Build hardware tomography circuits
#  Direct measurement on Y (no post-selection needed — deterministic decoder
#  routes all amplitude to the correct output).
# =============================================================================
def build_sweep_tomo(theta_rx, phi_rz, basis='Z',
                     theta_msg=THETA_MSG, varphi_msg=VARPHI_MSG):
    """
    Deterministic decoder with V(θ,φ) = Rz(φ)·Rx(θ) on C,
    with Pauli-basis tomography on output qubit Y.

    Circuit registers: C=q0, E=q1, R=q2, G=q3, M=q4, A=q5, Y=q6
    Classical: crY[0] for tomography measurement.
    """
    C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
    G=QuantumRegister(1,'G'); M=QuantumRegister(1,'M'); A=QuantumRegister(1,'A')
    Y=QuantumRegister(1,'Y')
    crY=ClassicalRegister(1,'crY')
    qc=QuantumCircuit(C,E,R,G,M,A,Y,crY)

    # Message prep
    qc.u(theta_msg, varphi_msg, 0.0, M[0])
    qc.swap(C[0], M[0]); qc.barrier()

    # Bell pairs
    qc.h(E[0]); qc.cx(E[0], M[0])
    qc.h(R[0]); qc.cx(R[0], G[0])
    qc.h(A[0]); qc.cx(A[0], Y[0]); qc.barrier()

    # Perturbation V(θ,φ) = Rz(φ)·Rx(θ) on C
    qc.rz(phi_rz, C[0])
    qc.rx(theta_rx, C[0]); qc.barrier()

    # Scrambling unitary on (C,E,R)
    qc.cz(C[0],R[0]); qc.cz(E[0],R[0]); qc.cz(C[0],E[0])
    qc.h(C[0]); qc.h(E[0]); qc.h(R[0])
    qc.cz(C[0],R[0]); qc.cz(C[0],E[0]); qc.cz(E[0],R[0]); qc.barrier()

    # Decoder U†  (first block)
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G)
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Unitary Transpose
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (A,Y)
    qc.rz(pi,A[0]); qc.rx(pi,A[0]); qc.rx(pi,Y[0])
    qc.swap(A[0],Y[0]); qc.rz(pi,A[0]); qc.barrier()

    # Unitary Conjugate again
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G) again
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Bell projection (measurement of syndrome)
    qc.cx(R[0],G[0]); qc.h(R[0]); qc.barrier()

    # Tomography rotation on Y then measure
    if basis == 'X': qc.h(Y[0])
    elif basis == 'Y': qc.sdg(Y[0]); qc.h(Y[0])
    qc.measure(Y[0], crY[0])
    return qc


# =============================================================================
#  STEP 2 — Pre-compute simulator predictions
# =============================================================================
print("=" * 65)
print("  Hardware Parameter Sweep — ibm_torino")
print("=" * 65)
print(f"\n  Grid: {N_THETA}×{N_PHI} = {N_THETA*N_PHI} points")
print(f"  Circuits: {N_THETA*N_PHI*3} (3 bases × grid)")
print(f"  Shots: {SHOTS} per circuit  ({N_THETA*N_PHI*3*SHOTS:,} total)")

print("\n  Computing simulator predictions ...")
qcm = QuantumCircuit(1); qcm.u(THETA_MSG, VARPHI_MSG, 0.0, 0)
psi_msg = Statevector.from_instruction(qcm)

F_sim = np.zeros((N_THETA, N_PHI))
for i, th in enumerate(THETAS):
    for j, ph in enumerate(PHIS):
        qc = build_sweep_tomo(th, ph, 'Z')
        qc_sv = qc.remove_final_measurements(inplace=False)
        psi = Statevector.from_instruction(qc_sv).data
        psi /= np.linalg.norm(psi)
        rho_f = psi[:, None] * psi.conj()[None, :]
        rho_Y = partial_trace(rho_f, [0, 1, 2, 3, 4, 5])
        F_sim[i, j] = float(state_fidelity(rho_Y, psi_msg))

print(f"  Simulator F range: [{F_sim.min():.4f}, {F_sim.max():.4f}]")
print(f"  Robust islands (F>0.8): {(F_sim>0.8).sum()}/{N_THETA*N_PHI}")
print(f"  Fragile valleys (F<0.4): {(F_sim<0.4).sum()}/{N_THETA*N_PHI}")


# =============================================================================
#  STEP 3 — Connect, transpile all 144 circuits
# =============================================================================
print(f"\n  Connecting to {BACKEND} ...")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"  [✓] Connected")

print(f"\n  Transpiling {N_THETA*N_PHI*3} circuits ...")
circuit_index = []  # (i_theta, j_phi, basis, transpiled_circuit)
for i, th in enumerate(THETAS):
    for j, ph in enumerate(PHIS):
        for basis in ['Z', 'X', 'Y']:
            qc = build_sweep_tomo(th, ph, basis)
            t  = transpile(qc, backend=backend,
                           optimization_level=3, seed_transpiler=SEED)
            circuit_index.append((i, j, basis, t))

# Report depth stats
depths = [t.depth() for _,_,_,t in circuit_index]
twoqs  = [sum(1 for _,qa,_ in t.data if len(qa)==2) for _,_,_,t in circuit_index]
print(f"  Transpiled depth:   mean={np.mean(depths):.0f}  "
      f"range=[{min(depths)},{max(depths)}]")
print(f"  Two-qubit gates:    mean={np.mean(twoqs):.0f}  "
      f"range=[{min(twoqs)},{max(twoqs)}]")


# =============================================================================
#  STEP 4 — Submit all circuits in one Batch job
# =============================================================================
print(f"\n  Submitting {len(circuit_index)} circuits in Batch ...")
jobs = {}
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = 'XX'
    for i, j, basis, t in circuit_index:
        key = f"{i}_{j}_{basis}"
        jobs[key] = sampler.run([t], shots=SHOTS)

print(f"  Submitted. Waiting for results ...")


# =============================================================================
#  STEP 5 — Collect and reconstruct
# =============================================================================
def extract_crY(result):
    """Extract counts for crY register from SamplerV2 result."""
    pub  = result[0]
    data = pub.data
    crY_arr = data.crY.array.flatten()
    counts = Counter(int(b) for b in crY_arr)
    return {'0': counts.get(0, 0), '1': counts.get(1, 0)}

def pauli_exp(counts):
    n0 = counts.get('0', 0); n1 = counts.get('1', 0)
    N  = n0 + n1
    return (n0 - n1) / N if N > 0 else 0.0

def reconstruct_dm(sx, sy, sz):
    X = np.array([[0,1],[1,0]], dtype=complex)
    Y = np.array([[0,-1j],[1j,0]], dtype=complex)
    Z = np.array([[1,0],[0,-1]], dtype=complex)
    rho = (np.eye(2) + sx*X + sy*Y + sz*Z) / 2
    ev, evec = np.linalg.eigh(rho)
    ev = np.maximum(ev, 0); ev /= ev.sum()
    return (evec * ev) @ evec.conj().T

def cp_ci(k, n, alpha=0.05):
    lo = beta_dist.ppf(alpha/2,   k,   n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k  ) if k < n else 1.0
    return float(lo), float(hi)

def bootstrap_F(cX, cY, cZ, rho_M, n=N_BOOTSTRAP):
    bsF = np.zeros(n)
    for b in range(n):
        def rs(c):
            keys = list(c.keys()); vals = np.array([c[k] for k in keys])
            N = vals.sum()
            if N == 0: return {'0':1,'1':1}
            new = RNG.multinomial(N, vals/N)
            return {k: int(v) for k, v in zip(keys, new)}
        sx = pauli_exp(rs(cX)); sy = pauli_exp(rs(cY)); sz = pauli_exp(rs(cZ))
        bsF[b] = float(state_fidelity(DensityMatrix(reconstruct_dm(sx,sy,sz)), rho_M))
    return bsF

print("  Collecting results ...")
counts_grid = {}    # (i,j,basis) -> {'0':n0,'1':n1}

for i, j, basis, _ in circuit_index:
    key = f"{i}_{j}_{basis}"
    print(f"    Waiting for {key} ...", end=" ", flush=True)
    result = jobs[key].result()
    counts_grid[(i, j, basis)] = extract_crY(result)
    total = sum(counts_grid[(i,j,basis)].values())
    print(f"done  ({total} shots)")

# Compute F_hw and reconstruct density matrices
print("\n  Reconstructing density matrices and computing fidelities ...")

qcm2 = QuantumCircuit(1); qcm2.u(THETA_MSG, VARPHI_MSG, 0.0, 0)
rho_M = DensityMatrix(Statevector.from_instruction(qcm2))

F_hw     = np.zeros((N_THETA, N_PHI))
F_hw_lo  = np.zeros((N_THETA, N_PHI))
F_hw_hi  = np.zeros((N_THETA, N_PHI))
F_hw_std = np.zeros((N_THETA, N_PHI))
delta_F  = np.zeros((N_THETA, N_PHI))   # F_hw - F_sim

grid_results = {}   # (i,j) -> full result dict

for i in range(N_THETA):
    for j in range(N_PHI):
        cZ = counts_grid[(i, j, 'Z')]
        cX = counts_grid[(i, j, 'X')]
        cY = counts_grid[(i, j, 'Y')]

        sx = pauli_exp(cX); sy = pauli_exp(cY); sz = pauli_exp(cZ)
        rho = reconstruct_dm(sx, sy, sz)
        F   = float(state_fidelity(DensityMatrix(rho), rho_M))
        bsF = bootstrap_F(cX, cY, cZ, rho_M)

        F_hw[i, j]     = F
        F_hw_lo[i, j]  = float(np.percentile(bsF, 2.5))
        F_hw_hi[i, j]  = float(np.percentile(bsF, 97.5))
        F_hw_std[i, j] = float(np.std(bsF))
        delta_F[i, j]  = F - F_sim[i, j]

        grid_results[(i, j)] = {
            'theta': float(THETAS[i]), 'phi': float(PHIS[j]),
            'F_hw': F, 'F_hw_lo': float(F_hw_lo[i,j]),
            'F_hw_hi': float(F_hw_hi[i,j]), 'F_hw_std': float(F_hw_std[i,j]),
            'F_sim': float(F_sim[i,j]),
            'delta_F': float(delta_F[i,j]),
            'sx': float(sx), 'sy': float(sy), 'sz': float(sz),
            'counts_Z': cZ, 'counts_X': cX, 'counts_Y': cY,
            'bootstrap_F': [float(v) for v in bsF],
        }

# =============================================================================
#  STEP 6 — Summary
# =============================================================================
print("\n" + "="*65)
print("  RESULTS SUMMARY")
print("="*65)
print(f"\n  Hardware F_msg:  mean={F_hw.mean():.4f}  "
      f"range=[{F_hw.min():.4f},{F_hw.max():.4f}]")
print(f"  Simulator F_msg: mean={F_sim.mean():.4f}  "
      f"range=[{F_sim.min():.4f},{F_sim.max():.4f}]")
print(f"  ΔF = F_hw - F_sim: mean={delta_F.mean():.4f}  "
      f"std={delta_F.std():.4f}")

print(f"\n  Qualitative agreement check:")
# Both have same robust/fragile classification?
hw_robust  = F_hw  > 0.7
sim_robust = F_sim > 0.7
agreement  = (hw_robust == sim_robust).sum()
print(f"  Robust island agreement (threshold F=0.7): "
      f"{agreement}/{N_THETA*N_PHI} points "
      f"({100*agreement/(N_THETA*N_PHI):.0f}%)")

from scipy.stats import spearmanr, pearsonr
rs, ps = spearmanr(F_hw.flatten(), F_sim.flatten())
rp, pp = pearsonr(F_hw.flatten(),  F_sim.flatten())
print(f"  Spearman r(F_hw, F_sim) = {rs:.4f}  p={ps:.3e}")
print(f"  Pearson  r(F_hw, F_sim) = {rp:.4f}  p={pp:.3e}")

# =============================================================================
#  STEP 7 — Save
# =============================================================================
class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray):    return o.tolist()
        return super().default(o)

# Convert tuple keys to strings for JSON
grid_results_str = {f"{k[0]}_{k[1]}": v for k, v in grid_results.items()}

payload = {
    "metadata": {
        "backend": BACKEND, "shots_per_circuit": SHOTS,
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "theta_msg": THETA_MSG, "varphi_msg": VARPHI_MSG,
        "n_theta": N_THETA, "n_phi": N_PHI,
        "thetas": THETAS.tolist(), "phis": PHIS.tolist(),
        "dd_sequence": "XX",
    },
    "F_hw":       F_hw.tolist(),
    "F_hw_lo":    F_hw_lo.tolist(),
    "F_hw_hi":    F_hw_hi.tolist(),
    "F_hw_std":   F_hw_std.tolist(),
    "F_sim":      F_sim.tolist(),
    "delta_F":    delta_F.tolist(),
    "spearman_r": float(rs), "spearman_p": float(ps),
    "pearson_r":  float(rp), "pearson_p":  float(pp),
    "grid_results": grid_results_str,
}

with open("hw_sweep_results.json", "w") as f:
    json.dump(payload, f, indent=2, cls=_Enc)
print("\n[✓] Saved to hw_sweep_results.json")
print("[✓] Run plot_hw_sweep.py for paper figures.")

qiskit_runtime_service._discover_account:WARNING:2026-03-22 14:42:03,257: Loading account with the given token. A saved account will not be used.


  Hardware Parameter Sweep — ibm_torino

  Grid: 6×8 = 48 points
  Circuits: 144 (3 bases × grid)
  Shots: 2000 per circuit  (288,000 total)

  Computing simulator predictions ...
  Simulator F range: [0.0001, 1.0000]
  Robust islands (F>0.8): 20/48
  Fragile valleys (F<0.4): 16/48

  Connecting to ibm_torino ...


qiskit_runtime_service.__init__:WARNING:2026-03-22 14:42:07,062: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-22 14:42:07,063: Using instance: CTCs, plan: open


  [✓] Connected

  Transpiling 144 circuits ...
  Transpiled depth:   mean=162  range=[157,165]
  Two-qubit gates:    mean=61  range=[61,61]

  Submitting 144 circuits in Batch ...
  Submitted. Waiting for results ...
    Waiting for 0_0_Z ... done  (2000 shots)
    Waiting for 0_0_X ... done  (2000 shots)
    Waiting for 0_0_Y ... done  (2000 shots)
    Waiting for 0_1_Z ... done  (2000 shots)
    Waiting for 0_1_X ... done  (2000 shots)
    Waiting for 0_1_Y ... done  (2000 shots)
    Waiting for 0_2_Z ... done  (2000 shots)
    Waiting for 0_2_X ... done  (2000 shots)
    Waiting for 0_2_Y ... done  (2000 shots)
    Waiting for 0_3_Z ... done  (2000 shots)
    Waiting for 0_3_X ... done  (2000 shots)
    Waiting for 0_3_Y ... done  (2000 shots)
    Waiting for 0_4_Z ... done  (2000 shots)
    Waiting for 0_4_X ... done  (2000 shots)
    Waiting for 0_4_Y ... done  (2000 shots)
    Waiting for 0_5_Z ... done  (2000 shots)
    Waiting for 0_5_X ... done  (2000 shots)
    Waiting for 0

[✓] fig_sweep1_comparison.pdf
[✓] fig_sweep2_delta.pdf
[✓] fig_sweep3_scatter.pdf
[✓] fig_sweep4_profiles.pdf
[✓] fig_sweep_panel.pdf

  PAPER-READY NUMBERS
  Backend:          ibm_torino
  Grid:             6×8 = 48 points
  Shots/circuit:    2000
  DD sequence:      XX

  Simulator F_msg:  mean=0.5990  std=0.3142  range=[0.0001,1.0000]
  Hardware F_msg:   mean=0.5995  std=0.1865  range=[0.2057,0.8174]
  Mean ΔF:          0.0005  (bias)
  Std  ΔF:          0.1404  (scatter)

  Agreement at F>0.5: 43/48 (90%)
  Agreement at F>0.7: 42/48 (88%)
  Agreement at F>0.8: 34/48 (71%)

  Spearman r(F_hw, F_sim) = 0.9497  p=7.923e-25
  Pearson  r(F_hw, F_sim) = 0.9709  p=3.296e-30


In [2]:
"""
Parameter sweep figures — hardware vs simulator
================================================
Input:  hw_sweep_results.json
Output: fig_sweep1_heatmaps.pdf/.png
        fig_sweep2_scatter.pdf/.png
        fig_sweep3_profiles.pdf/.png

Usage:
    python3 plot_hw_sweep.py

Edit the two paths at the top to match your setup.
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ── Paths — edit these ────────────────────────────────────────────────────────
JSON_PATH = 'hw_sweep_results.json'   # path to your JSON file
OUT_DIR   = './'                      # where to save the figures

# ── Load data ─────────────────────────────────────────────────────────────────
with open(JSON_PATH) as f:
    D = json.load(f)

F_hw   = np.array(D['F_hw'])
F_sim  = np.array(D['F_sim'])
dF     = np.array(D['delta_F'])
F_lo   = np.array(D['F_hw_lo'])
F_hi   = np.array(D['F_hw_hi'])
rs     = D['spearman_r']
ps     = D['spearman_p']
rp     = D['pearson_r']
thetas = np.array(D['metadata']['thetas'])
phis   = np.array(D['metadata']['phis'])
shots  = D['metadata']['shots_per_circuit']
N_th   = len(thetas)
N_ph   = len(phis)

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   9.5,
    'figure.dpi':        180,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'pdf.fonttype':      42,
})

# ── Helper: annotate heatmap cells ────────────────────────────────────────────
def annotate_cells(ax, data, fmt='.2f', fontsize=8.5, threshold=0.5):
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            color = 'white' if data[i, j] < threshold else '#222222'
            ax.text(j, i, f'{data[i, j]:{fmt}}',
                    ha='center', va='center',
                    fontsize=fontsize, color=color)

# ── Tick label helpers ────────────────────────────────────────────────────────
phi_labels   = [f'{p/np.pi:.1f}$\\pi$' for p in phis]
theta_labels = [f'{t/np.pi:.2f}$\\pi$' for t in thetas]

# =============================================================================
#  FIG 1 — Side-by-side heatmaps: simulator, hardware, discrepancy
# =============================================================================
fig1, axes = plt.subplots(1, 3, figsize=(14, 4.2),
                           gridspec_kw={'width_ratios': [1, 1, 1]})

cmap_F  = 'RdYlGn'
cmap_dF = 'RdBu_r'
absmax  = np.abs(dF).max()

panels = [
    (axes[0], F_sim, cmap_F,  0, 1,       '$F_{\\rm msg}$',
     '(a) Simulator $F_{\\rm msg}(\\theta,\\phi)$\n(noiseless)'),
    (axes[1], F_hw,  cmap_F,  0, 1,       '$F_{\\rm msg}$',
     f'(b) Hardware $F_{{\\rm msg}}(\\theta,\\phi)$\n(ibm_torino, {shots} shots/pt)'),
    (axes[2], dF,    cmap_dF, -absmax, absmax, '$\\Delta F$',
     f'(c) $\\Delta F = F_{{\\rm hw}} - F_{{\\rm sim}}$\n'
     f'mean={dF.mean():+.3f},  std={dF.std():.3f}'),
]

for ax, data, cmap, vmin, vmax, cblabel, title in panels:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                   aspect='auto', origin='upper')
    thresh = 0.5 if cmap == cmap_F else 0.0
    fmt    = '.2f' if cmap == cmap_F else '+.2f'
    annotate_cells(ax, data, fmt=fmt, threshold=thresh)
    ax.set_title(title, pad=8)
    ax.set_xticks(range(N_ph))
    ax.set_xticklabels(phi_labels, fontsize=8.5, rotation=45, ha='right')
    ax.set_yticks(range(N_th))
    ax.set_yticklabels(theta_labels, fontsize=9)
    ax.set_xlabel('$\\phi$')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.06)
    cb  = fig1.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=8.5)
    cb.set_label(cblabel, fontsize=9)

axes[0].set_ylabel('$\\theta$')

fig1.suptitle(
    f'Parameter sweep: $F_{{\\rm msg}}(\\theta,\\phi)$ — ibm_torino vs noiseless simulator\n'
    f'Grid: {N_th}$\\times${N_ph},  '
    f'Spearman $r={rs:.4f}$ ($p={ps:.1e}$),  '
    f'Pearson $r={rp:.4f}$',
    fontsize=11, y=1.02
)
fig1.tight_layout()
fig1.savefig(OUT_DIR + 'fig_sweep1_heatmaps.pdf', bbox_inches='tight', dpi=200)
fig1.savefig(OUT_DIR + 'fig_sweep1_heatmaps.png', bbox_inches='tight', dpi=200)
plt.close(fig1)
print('[✓] fig_sweep1_heatmaps.pdf/.png')

# =============================================================================
#  FIG 2 — Scatter plot F_hw vs F_sim, coloured by theta
# =============================================================================
fig2, ax = plt.subplots(figsize=(5.5, 5.0))

cmap_sc    = plt.cm.plasma
theta_norm = (thetas - thetas.min()) / (thetas.max() - thetas.min())

for i, tn in enumerate(theta_norm):
    ci_y = (F_hi[i] - F_lo[i]) / 2
    ax.errorbar(F_sim[i], F_hw[i],
                yerr=ci_y,
                fmt='o', color=cmap_sc(tn), ms=7,
                elinewidth=1.0, capsize=3, capthick=1.0,
                ecolor=cmap_sc(tn), alpha=0.85, zorder=3)

# y = x reference line
ax.plot([0, 1], [0, 1], 'k--', lw=1.0, alpha=0.4,
        label='$y=x$')

# linear fit
c1, c0 = np.polyfit(F_sim.flatten(), F_hw.flatten(), 1)
x_ref  = np.linspace(0, 1, 100)
ax.plot(x_ref, c0 + c1 * x_ref, color='#C0392B', lw=1.8,
        label=f'Linear fit')

# colorbar for theta
sm = plt.cm.ScalarMappable(
    cmap=cmap_sc, norm=plt.Normalize(thetas.min(), thetas.max()))
sm.set_array([])
cb = fig2.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('$\\theta$ (rad)', fontsize=10)
cb.ax.tick_params(labelsize=9)
cb.set_ticks([0, np.pi / 2, np.pi])
cb.set_ticklabels(['$0$', '$\\pi/2$', '$\\pi$'])

ax.set_xlabel('$F_{\\rm sim}(\\theta,\\phi)$')
ax.set_ylabel('$F_{\\rm hw}(\\theta,\\phi)$')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9, framealpha=0.9, loc='upper left')
ax.yaxis.grid(True, alpha=0.25, zorder=0)
ax.xaxis.grid(True, alpha=0.25, zorder=0)

fig2.tight_layout()
fig2.savefig(OUT_DIR + 'fig_sweep2_scatter.pdf', bbox_inches='tight', dpi=200)
fig2.savefig(OUT_DIR + 'fig_sweep2_scatter.png', bbox_inches='tight', dpi=200)
plt.close(fig2)
print('[✓] fig_sweep2_scatter.pdf/.png')

# =============================================================================
#  FIG 3 — Line profiles: F_hw vs F_sim at each fixed theta
# =============================================================================
fig3, axes3 = plt.subplots(2, 3, figsize=(13, 6.5),
                             sharey=True, sharex=True)
axes3 = axes3.flatten()

C_SIM = '#2980B9'
C_HW  = '#C0392B'

for i, th in enumerate(thetas):
    ax = axes3[i]

    ax.plot(phis, F_sim[i], color=C_SIM, lw=2.0, ls='--',
            label='Simulator', zorder=3)
    ax.plot(phis, F_hw[i],  color=C_HW,  lw=2.0, marker='o',
            ms=5.5, label='Hardware', zorder=4)
    ax.fill_between(phis, F_lo[i], F_hi[i],
                    color=C_HW, alpha=0.18, zorder=2,
                    label='95% CI (hardware)')

    ax.set_title(f'$\\theta = {th/np.pi:.2f}\\pi$', fontsize=11)
    ax.set_ylim(-0.05, 1.10)
    ax.axhline(1.0, color='#27AE60', lw=0.8, ls=':', alpha=0.5)
    ax.yaxis.grid(True, alpha=0.25, zorder=0)
    ax.set_xticks(phis[::2])
    ax.set_xticklabels(
        [f'{p/np.pi:.1f}$\\pi$' for p in phis[::2]],
        fontsize=9, rotation=30, ha='right')

    if i % 3 == 0:
        ax.set_ylabel('$F_{\\rm msg}$')
    if i >= 3:
        ax.set_xlabel('$\\phi$')
    if i == 0:
        ax.legend(fontsize=9.5, framealpha=0.9, loc='lower right')

fig3.suptitle(
    'Fidelity profiles: hardware (red) vs simulator (blue dashed)\n'
    'Shaded band = 95% bootstrap CI on hardware',
    fontsize=11, y=1.01
)
fig3.tight_layout()
fig3.savefig(OUT_DIR + 'fig_sweep3_profiles.pdf', bbox_inches='tight', dpi=200)
fig3.savefig(OUT_DIR + 'fig_sweep3_profiles.png', bbox_inches='tight', dpi=200)
plt.close(fig3)
print('[✓] fig_sweep3_profiles.pdf/.png')

# =============================================================================
#  Paper-ready summary
# =============================================================================
print(f"\n{'='*60}")
print("  PAPER-READY NUMBERS")
print(f"{'='*60}")
print(f"  Grid:            {N_th}x{N_ph} = {N_th*N_ph} points")
print(f"  Shots/circuit:   {shots}")
print(f"  F_sim range:     [{F_sim.min():.4f}, {F_sim.max():.4f}]  mean={F_sim.mean():.4f}")
print(f"  F_hw  range:     [{F_hw.min():.4f}, {F_hw.max():.4f}]  mean={F_hw.mean():.4f}")
print(f"  Mean dF:         {dF.mean():+.4f}")
print(f"  Std  dF:         {dF.std():.4f}")
print(f"  Spearman r:      {rs:.4f}  (p={ps:.2e})")
print(f"  Pearson  r:      {rp:.4f}")
for thresh in [0.5, 0.7]:
    agree = ((F_hw > thresh) == (F_sim > thresh)).sum()
    print(f"  Island agree (F>{thresh}): {agree}/{N_th*N_ph} "
          f"({100*agree/(N_th*N_ph):.0f}%)")

[✓] fig_sweep1_heatmaps.pdf/.png
[✓] fig_sweep2_scatter.pdf/.png
[✓] fig_sweep3_profiles.pdf/.png

  PAPER-READY NUMBERS
  Grid:            6x8 = 48 points
  Shots/circuit:   2000
  F_sim range:     [0.0001, 1.0000]  mean=0.5990
  F_hw  range:     [0.2057, 0.8174]  mean=0.5995
  Mean dF:         +0.0005
  Std  dF:         0.1404
  Spearman r:      0.9497  (p=7.92e-25)
  Pearson  r:      0.9709
  Island agree (F>0.5): 43/48 (90%)
  Island agree (F>0.7): 42/48 (88%)
